In [1]:
"""
Mode Collapse Diagnostic - U2R
====================================================================
Purpose: Loss curves alone can look "stable" even when a GAN has
partially collapsed (generator producing low-diversity samples that
still fool the critic on average). This script checks diversity
directly by comparing pairwise similarity within:
  (a) real U2R samples
  (b) WCGAN-synthetic U2R samples

If synthetic samples are FAR more similar to each other than real
samples are to each other, that's a mode collapse red flag -- even
if the loss curve looked fine.

Requires: pandas, numpy, scikit-learn (pip install scikit-learn if needed)

Author: Vibha J | GAN Implementation Lead
====================================================================
"""

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# ============================================================
# CONFIG -- EDIT THESE PATHS
# ============================================================
REAL_PATH = "./data/gan_input_U2R.csv"
SYNTHETIC_PATH = "./wcgan_outputs/synthetic_U2R.csv"


def pairwise_stats(data, label):
    """Computes average pairwise cosine similarity and euclidean
    distance, excluding self-comparisons (diagonal)."""
    n = data.shape[0]

    cos_sim = cosine_similarity(data)
    np.fill_diagonal(cos_sim, np.nan)
    avg_cos_sim = np.nanmean(cos_sim)

    eucl_dist = euclidean_distances(data)
    np.fill_diagonal(eucl_dist, np.nan)
    avg_eucl_dist = np.nanmean(eucl_dist)

    print(f"\n--- {label} (n={n}) ---")
    print(f"Avg pairwise cosine similarity: {avg_cos_sim:.4f}  (higher = more similar/less diverse)")
    print(f"Avg pairwise euclidean distance: {avg_eucl_dist:.4f}  (lower = more similar/less diverse)")

    return avg_cos_sim, avg_eucl_dist


def nearest_neighbor_check(data, label, k=1):
    """For each sample, finds its closest OTHER sample. If many
    synthetic samples have near-zero distance to their nearest
    neighbor, that's a strong sign of collapse (near-duplicate
    generation)."""
    eucl_dist = euclidean_distances(data)
    np.fill_diagonal(eucl_dist, np.inf)
    nn_dist = eucl_dist.min(axis=1)

    print(f"\n--- {label}: Nearest-neighbor distance ---")
    print(f"Min: {nn_dist.min():.4f}  Median: {np.median(nn_dist):.4f}  Max: {nn_dist.max():.4f}")

    near_dup_threshold = np.percentile(nn_dist, 5) * 0.1  # heuristic: very tight cluster
    n_near_dupes = (nn_dist < near_dup_threshold).sum()
    if n_near_dupes > 0:
        print(f"WARNING: {n_near_dupes} samples have a near-duplicate neighbor "
              f"(distance < {near_dup_threshold:.4f}). Possible collapse.")
    else:
        print("No near-duplicate clusters detected at this threshold.")

    return nn_dist


if __name__ == "__main__":
    real_df = pd.read_csv(REAL_PATH)
    synth_df = pd.read_csv(SYNTHETIC_PATH)

    real_data = real_df.values.astype(np.float64)
    synth_data = synth_df.values.astype(np.float64)

    print("=" * 60)
    print("MODE COLLAPSE DIAGNOSTIC - U2R")
    print("=" * 60)

    real_cos, real_eucl = pairwise_stats(real_data, "REAL U2R")
    synth_cos, synth_eucl = pairwise_stats(synth_data, "SYNTHETIC U2R (WCGAN)")

    nearest_neighbor_check(real_data, "REAL U2R")
    nearest_neighbor_check(synth_data, "SYNTHETIC U2R (WCGAN)")

    print("\n" + "=" * 60)
    print("VERDICT")
    print("=" * 60)
    cos_ratio = synth_cos / real_cos if real_cos != 0 else float("inf")
    eucl_ratio = synth_eucl / real_eucl if real_eucl != 0 else 0

    print(f"Synthetic-to-real cosine similarity ratio: {cos_ratio:.2f}x")
    print(f"Synthetic-to-real euclidean distance ratio: {eucl_ratio:.2f}x")
    print()
    if cos_ratio > 1.5 or eucl_ratio < 0.5:
        print("LIKELY MODE COLLAPSE: synthetic samples are noticeably more")
        print("similar to each other than real samples are. Consider:")
        print("  - Increasing LATENT_DIM for more input diversity")
        print("  - Reducing N_CRITIC (currently 5) slightly")
        print("  - Training for fewer epochs (check if collapse happens late)")
    else:
        print("NO STRONG COLLAPSE SIGNAL: synthetic sample diversity is")
        print("comparable to real sample diversity. WCGAN output looks healthy.")

MODE COLLAPSE DIAGNOSTIC - U2R

--- REAL U2R (n=198) ---
Avg pairwise cosine similarity: 0.8866  (higher = more similar/less diverse)
Avg pairwise euclidean distance: 4.3118  (lower = more similar/less diverse)

--- SYNTHETIC U2R (WCGAN) (n=198) ---
Avg pairwise cosine similarity: 0.9127  (higher = more similar/less diverse)
Avg pairwise euclidean distance: 3.4906  (lower = more similar/less diverse)

--- REAL U2R: Nearest-neighbor distance ---
Min: 0.0007  Median: 0.0640  Max: 2.8863

--- SYNTHETIC U2R (WCGAN): Nearest-neighbor distance ---
Min: 0.0074  Median: 0.4448  Max: 1.5072
No near-duplicate clusters detected at this threshold.

VERDICT
Synthetic-to-real cosine similarity ratio: 1.03x
Synthetic-to-real euclidean distance ratio: 0.81x

NO STRONG COLLAPSE SIGNAL: synthetic sample diversity is
comparable to real sample diversity. WCGAN output looks healthy.
